## QSDC Demonstration

The below cell demonstrates the QSDC protocol. Observe that when eavesdropping is detected, the message becomes corrupted - this is due to the No-Cloning Theorem and the inherent neature of measuring an eigenvalue. You can't "peek" at a value without leaving a trace, or collapsing a qubit to an eigenstate of the chosen basis.

In [100]:
import numpy as np

def gaussian_elimination_gf2(matrix):
    """
    Transforms a matrix into Systematic Form [I_k | Q] over GF(2).
    Returns (systematic_matrix, success_flag).
    """
    m = matrix.copy() % 2
    k, n = m.shape
    
    for r in range(k):
        # 1. Find pivot row
        pivot = -1
        for i in range(r, k):
            if m[i, r] == 1:
                pivot = i
                break
        
        # If no pivot is found in this column, it cannot form a leading identity matrix I_k
        if pivot == -1:
            return None, False
            
        # Swap current row with pivot row
        if pivot != r:
            m[[r, pivot]] = m[[pivot, r]]
            
        # 2. Eliminate other rows
        for i in range(k):
            if i != r and m[i, r] == 1:
                m[i] = (m[i] + m[r]) % 2
                
    return m, True

def generate_systematic_keypair(k, n, t):
    """
    Generates a McEliece keypair where G is strictly systematic: [I_k | Q]
    """
    while True:
        # Generate a random baseline generator matrix G
        G_raw = np.random.randint(0, 2, size=(k, n))
        
        # Force G into systematic form [I_k | Q] using Gaussian Elimination
        G_sys, success = gaussian_elimination_gf2(G_raw)
        if not success:
            continue  # Abort and restart key generation (matches Classic McEliece spec)
            
        # Generate a random invertible k x k scrambling matrix S
        while True:
            S = np.random.randint(0, 2, size=(k, k))
            if int(np.round(np.linalg.det(S))) % 2 != 0:
                break
                
        # Generate an n x n permutation matrix P
        P = np.eye(n, dtype=int)[np.random.permutation(n)]
        
        # Compute Public Key: G_prime = (S * G_sys * P) mod 2
        G_prime = np.dot(S, G_sys) % 2
        G_prime = np.dot(G_prime, P) % 2
        
        # Note: In the final Classic McEliece standard, the public key is strictly 
        # compressed to just the 'Q' matrix part since 'I_k' is public knowledge.
        return (S, G_sys, P), (G_prime, t)

def decrypt_systematic(ciphertext, private_key):
    """
    Deterministic decryption using the systematic properties.
    """
    S, G_sys, P = private_key
    
    # 1. Reverse Permutation
    c_hat = np.dot(ciphertext, P.T) % 2
    
    # 2. Error Correction (Simulated placeholder)
    c_clean = c_hat  
    
    # 3. Extract Message from Codeword
    # Because G_sys is [I_k | Q], the first k bits of c_clean are exactly the scrambled message!
    k = G_sys.shape[0]
    m_prime = c_clean[:k]
    
    # 4. Reverse Scrambling
    S_inv = np.round(np.linalg.inv(S)).astype(int) % 2
    message = np.dot(m_prime, S_inv) % 2
    
    return message

# --- Verification Run ---
k, n, t = 7, 11, 1  
msg = np.array([1, 0, 1, 1, 0, 1, 0])

priv, pub = generate_systematic_keypair(k, n, t)

# Encrypt
G_prime, _ = pub
c_pure = np.dot(msg, G_prime) % 2
e = np.zeros(n, dtype=int)
e[np.random.choice(n, t, replace=False)] = 1
ciphertext = (c_pure + e) % 2

# Decrypt (assuming 0 errors for the simulation step)
# This will now succeed 100% of the time because G_sys[:k] is guaranteed to be I_k.
decrypted_msg = decrypt_systematic(c_pure, priv) 

print(f"Systematic G Matrix Structure:\n{priv[1]}")
print(f"Original:  {msg}")
print(f"Decrypted: {decrypted_msg}")


Systematic G Matrix Structure:
[[1 0 0 0 0 0 0 0 1 0 0]
 [0 1 0 0 0 0 0 1 1 1 0]
 [0 0 1 0 0 0 0 1 0 1 0]
 [0 0 0 1 0 0 0 0 1 0 0]
 [0 0 0 0 1 0 0 0 1 0 0]
 [0 0 0 0 0 1 0 0 1 1 1]
 [0 0 0 0 0 0 1 1 0 1 1]]
Original:  [1 0 1 1 0 1 0]
Decrypted: [1 0 1 1 0 1 0]


In [ ]:
import pyspx.shake256_128f as sphincs

# 1. Generate a key pair
# SPHINCS+ features exceptionally small keys (e.g., 32-byte public key)
public_key, secret_key = sphincs.generate_keypair()

print(f"Public Key size: {len(public_key)} bytes")
print(f"Secret Key size: {len(secret_key)} bytes")

# 2. Sign a message
message = b"This is a post-quantum secure message."
signature = sphincs.sign(message, secret_key)

# Warning: SPHINCS+ signatures are very large (approx. 17 KB for this variant)
print(f"Signature size: {len(signature)} bytes")

# 3. Verify the signature
is_valid = sphincs.verify(message, signature, public_key)
print(f"Is signature valid? {is_valid}")

# 4. Attempt verification with altered data
tampered_message = b"This is a post-quantum secure message!"
is_valid_tampered = sphincs.verify(tampered_message, signature, public_key)
print(f"Is tampered signature valid? {is_valid_tampered}")
